# Transform Order Payments Data
1. filter our invalid rows i.e rows with null order_id and payment sequential or duplicated rows with order_id and payment_sequential
2. standardize the payment_type value into lowercase
3. write the transformed data into silver tables

In [0]:
#Imports
from pyspark.sql.functions import col,trim, lower

In [0]:
order_payments_df = spark.read.table("olist_catalog.bronze.order_payments")

### Step 1 - filter our invalid rows i.e rows with null order_id and payment sequential or duplicated rows with order_id and payment_sequential

In [0]:
order_payments_valid_df=(
    order_payments_df.filter(col("order_id").isNotNull() & col("payment_sequential").isNotNull()).dropDuplicates(subset=["order_id","payment_sequential"])
)

### Step2 - standardize the payment_type value into lowercase

In [0]:
order_payments_final_df = order_payments_valid_df.withColumn('payment_type',lower(trim(col("payment_type"))))

### Step3 - standardize the payment_type value into lowercase

In [0]:
(
    order_payments_final_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("olist_catalog.silver.order_payments")
)

In [0]:
%sql
select * from olist_catalog.silver.order_payments